## 1. Setup & imports

Import required libraries and load the ESA SNAP `esa_snappy` Python bindings

In [ ]:
# Import required modules
import numpy as np
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import pandas as pd
import folium
import math
import os
import re
import json
import requests
import geopandas as gpd
import zipfile
import shlex
import shapely as shp
import rasterio
import config
import xml.etree.ElementTree as ET
from rasterio.warp import transform_bounds
from datetime import datetime, timedelta
from subprocess import Popen, PIPE, STDOUT
from glob import iglob, glob
from scipy import ndimage
from termcolor import colored

try:
    import esa_snappy
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "esa_snappy is not installed in this environment. This notebook requires "
        "ESA SNAP with its Python bindings configured (via snappy-conf) on the same "
        "machine running this kernel - see https://step.esa.int/main/toolboxes/snap/. "
        "On Windows, running SNAP + esa_snappy under WSL is recommended."
    ) from e

## 2. Configuration & parameters

File paths, the conflict baseline date and its (asymmetric) search window, Sentinel-1/SNAP processing parameters, and the AOI polygon (loaded from GeoJSON).

In [ ]:
# File paths
input_dir = './data/input/s1/'
prep_dir = './data/output/preprocessed/'
results_dir = './data/output/results/'  # final damage-assessment products. One subfolder per date triplet
graph_file_path = './data/graphs/insar-prep.xml'

# Baseline date
baseline = '2026-06-24' # Event start date
# Event end date. Leave as 'None' for shorter-term events/analysis.
conflict_end = None 
# Search window: look far enough back to catch >=2 pre-baseline acquisitions on the same relative
# orbit (Sentinel-1's repeat cycle is ~12 days), since damage detection needs a pre-event "baseline"
# coherence pair (pre1/pre2) in addition to the co-event pair (pre2/post).
days_before = 30
days_after = 14
# Convert baseline (and, if given, the conflict end) to datetime objects
baseline_date = datetime.strptime(baseline, '%Y-%m-%d')
post_anchor_date = datetime.strptime(conflict_end, '%Y-%m-%d') if conflict_end else baseline_date
# Calculate the search window bounds
before = baseline_date - timedelta(days_before)
after = post_anchor_date + timedelta(days_after)
# Convert the results back to strings
before_str = before.strftime('%Y-%m-%d')
after_str = after.strftime('%Y-%m-%d')

# Sentinel-1 search parameters
data_collection = 'SENTINEL-1'
product_type = 'IW_SLC__1S'

# Processing parameters
speckle_filter = 'Lee' 
speckle_windowsize = '3' 
proj = 'EPSG:3395'  # WGS 84 / World Mercator 
# DEM source for terrain correction and geocoding.
dem = 'Copernicus 30m Global DEM'

# AOI polygon, drawn at http://geojson.io and saved to data/aoi/
aoi_path = './data/aoi/caracas.geojson'
aoi = gpd.read_file(aoi_path).geometry.iloc[0]
aoi_centroid = aoi.centroid  # Used to center the preview/damage maps on whichever AOI is loaded
subset_polygon = aoi.wkt  # WKT string passed to the SNAP graph to subset each scene to the AOI
out_format = 'GeoTIFF'

# Get format suffix
if out_format == 'GeoTIFF': suffix = '.tif'
elif out_format == 'BEAM-DIMAP': suffix = '.dim'
else: print('Please select either GeoTIFF or BEAM-DIMAP')

## 3. Helper functions

Small utilities for building and running the SNAP `gpt` command line.

In [ ]:
# Adds a quotation mark to string for use in cmd needed to run the SNAP graph
def dq(s):
    return '"' + s + '"'

# Call a terminal and execute a command and print the terminal output in the output cell.
def execute(command):
    # Merge stderr into stdout otherwise gpt's actual error messages never show up 
    with Popen(shlex.split(command), stdout=PIPE, stderr=STDOUT, bufsize=1, universal_newlines=True) as p:
        while True:
            line = p.stdout.readline()
            if not line:
                break
            print(line)    
        exit_code = p.wait()
    return exit_code

## 4. Authenticate with CDSE

Log in to the Copernicus Data Space Ecosystem and retrieve an access token for the catalog/download APIs.

In [ ]:
# Retrieve Copernicus access token
def get_access_token(username: str, password: str) -> str:
    data = {
        "client_id": "cdse-public",
        "username": username,
        "password": password,
        "grant_type": "password",
        }
    try:
        r = requests.post("https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
        data=data,
        )
        r.raise_for_status()
    except Exception as e:
        raise Exception(
            f"Access token creation failed. Reponse from the server was: {r.json()}"
            )
    return r.json()["access_token"]
        
access_token = get_access_token(config.username, config.password)

## 5. Query the Copernicus catalog

Search CDSE for Sentinel-1 IW SLC scenes over the AOI within the search window, build a GeoDataFrame of candidate products with their relative orbit and acquisition date, keep only scenes whose footprint fully contains the AOI, and auto-select the relative orbit with the most AOI-containing scenes.

In [ ]:
# Build the request and get the results
catalog_response = requests.get(
    f"https://catalogue.dataspace.copernicus.eu/odata/v1/Products?"
    f"$filter=Collection/Name eq '{data_collection}' "
    f"and OData.CSC.Intersects(area=geography'SRID=4326;{aoi}') "
    f"and Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' "
    f"and att/OData.CSC.StringAttribute/Value eq '{product_type}') "
    f"and ContentDate/Start ge {before_str}T00:00:00.000Z "
    f"and ContentDate/Start le {after_str}T00:00:00.000Z&$expand=Attributes"
    f"&$orderby=ContentDate/Start asc"
    f"&$top=1000" 
).json()

results = pd.DataFrame.from_dict(catalog_response['value'])

# Function to extract relative orbit number from the 'Attributes' column
def extract_relative_orbit_number(attributes):
    for attribute in attributes:
        if attribute['Name'] == 'relativeOrbitNumber' and attribute['@odata.type'] == '#OData.CSC.IntegerAttribute':
            return attribute['Value']
    return None

# Apply the function to the 'Attributes' column to create a new 'RelativeOrbitNumber' column
results['RelativeOrbitNumber'] = results['Attributes'].apply(extract_relative_orbit_number)

# Function to convert geofootprint to a polygon
def geofootprint_to_polygon(geofootprint):
    # Get the coordinates
    coords = geofootprint['coordinates'][0]
    # Convert to a polygon
    polygon = shp.geometry.Polygon(coords)
    return polygon

# Create GeoDataFrame with the required columns
results_gdf = gpd.GeoDataFrame(results[['Id', 'Name', 'RelativeOrbitNumber']], 
                               geometry=results['GeoFootprint'].apply(geofootprint_to_polygon), 
                               crs='EPSG:4326')

# Function to extract the acquisition date from the 'Name' column
def extract_acquisition_date(name):
    # Extract the date part from the filename
    date_str = name.split('1SDV_')[1][:8]
    return pd.to_datetime(date_str)

# Apply the function to the 'Name' column to create a new 'AcquisitionDate' column
results_gdf['AcquisitionDate'] = results_gdf['Name'].apply(extract_acquisition_date)

# The CDSE catalog filter above only checks for *intersection* with the AOI 
# Keep only scenes that fully contain it.
before_containment_count = len(results_gdf)
results_gdf = results_gdf[results_gdf.geometry.contains(aoi)].reset_index(drop=True)
print(f"{len(results_gdf)} of {before_containment_count} candidate scenes fully contain the AOI "
      f"(the rest only partially intersect it and were dropped)")

# Auto-select the relative orbit: pick whichever one has the most scenes fully containing the AOI
orbit_counts = results_gdf['RelativeOrbitNumber'].value_counts()
relative_orbit = int(orbit_counts.idxmax())
print(f"Relative orbits covering the AOI (scene count): {orbit_counts.to_dict()}")
print(f"Auto-selected relative orbit {relative_orbit}")

# Show GeoDataFrame
results_gdf

## 6. Preview candidate scenes

Plot the AOI and the first ten candidate scenes on an interactive map to sanity-check coverage before picking a triplet.

In [ ]:
# Create a Folium map centered over AOI
f = folium.Figure(width=2000, height=1000)
map = folium.Map([aoi_centroid.y, aoi_centroid.x],
                  zoom_start=8,
                  tiles='cartodbdark_matter').add_to(f)

# Translate aoi to geojson for quick addition to Folium maps
geojson = folium.GeoJson(data=gpd.GeoSeries(aoi).to_json(), name='AOI')
geojson.add_to(map)

# Convert date-time objects to string format in 'results_gdf' before plotting
results_gdf['AcquisitionDate'] = results_gdf['AcquisitionDate'].dt.strftime('%Y-%m-%d')

# Add each candidate scene as its own layer
scene_colors = ['red', 'blue', 'green', 'purple', 'orange',
                 'darkred', 'cadetblue', 'darkgreen', 'darkpurple', 'pink']
for i, (_, row) in enumerate(results_gdf.iloc[:10].iterrows()):
    color = scene_colors[i % len(scene_colors)]
    layer_name = f"{row['AcquisitionDate']} (orbit {row['RelativeOrbitNumber']})"
    folium.GeoJson(
        data=gpd.GeoSeries(row.geometry).to_json(),
        name=layer_name,
        style_function=lambda x, color=color: {'color': color, 'opacity': 1, 'fill': False},
        tooltip=f"{row['Name']}<br>Id: {row['Id']}",
    ).add_to(map)

folium.LayerControl().add_to(map)

# Save Folium map and open it manually.
preview_map_path = os.path.abspath('./data/output/candidate_scenes_preview.html')
map.save(preview_map_path)
print(f"Map saved. Open it in your browser from: {preview_map_path}")

## 7. Select the pre1/pre2/post image triplet

Pick three scenes on the auto-selected relative orbit: two pre-baseline images (forming the "baseline" coherence pair, used to characterize non-damage decorrelation) and the closest post-baseline image (paired with the closer pre-baseline image to form the "co-event" coherence pair).

In [ ]:
def select_images(df, baseline, relative_orbit, post_anchor=None):
    # Convert baseline (and the post-event anchor date, if given) to datetime for comparison
    baseline_date = pd.to_datetime(baseline)
    post_anchor_date = pd.to_datetime(post_anchor) if post_anchor else baseline_date

    # Ensure 'AcquisitionDate' is in datetime format for comparison
    df['AcquisitionDate'] = pd.to_datetime(df['AcquisitionDate'])

    # Restrict to the configured relative orbit
    df = df[df['RelativeOrbitNumber'] == relative_orbit]

    selected_images_list = []  # Initialize an empty list to store selected rows

    # Sort by date
    sorted_group = df.sort_values('AcquisitionDate')

    # All images on or before the baseline (candidates for the pre-event baseline pair)
    pre = sorted_group[sorted_group['AcquisitionDate'] <= baseline_date]
    # Images after the baseline (co-event image, paired with the closest pre-event image)
    post = sorted_group[sorted_group['AcquisitionDate'] > baseline_date]

    # Need at least two pre-baseline acquisitions
    if len(pre) >= 2 and not post.empty:
        selected_image_pre1 = pre.iloc[-2]   # earlier pre-event image (baseline-pair anchor)
        selected_image_pre2 = pre.iloc[-1]   # closest pre-event image (shared by both pairs)
        # Post-event image closest to the anchor date
        post_idx = (post['AcquisitionDate'] - post_anchor_date).abs().idxmin()
        selected_image_post = post.loc[post_idx]

        selected_images_list.extend([selected_image_pre1, selected_image_pre2, selected_image_post])

    # Convert the list of selected images to a DataFrame
    selected_images = pd.DataFrame(selected_images_list)
    return selected_images

selected_images = select_images(results_gdf, baseline, relative_orbit, post_anchor=conflict_end)

# Print the details of the selected images
if not selected_images.empty:
    labels = ['Pre-event (baseline pair)', 'Pre-event (closest to baseline)', 'Post-event (co-event)']
    print("Recommended Image Triplet:")
    for label, (idx, row) in zip(labels, selected_images.iterrows()):
        print(f"[{label}] ID: {row['Id']}, Name: {row['Name']}, Relative Orbit Number: {row['RelativeOrbitNumber']}, Acquisition Date: {row['AcquisitionDate'].strftime('%Y-%m-%d')}")
else:
    print(f"No suitable image triplet found on relative orbit {relative_orbit}.")

## 8. Download the selected products

Download the three selected Sentinel-1 SLC products (pre1, pre2, post) from CDSE.

In [ ]:
# Specify download directory
# Create the directory if it does not exist
output_dir = "./data/input/s1"
os.makedirs(output_dir, exist_ok=True)

# Function to download product
def download_product(product_id, access_token, output_dir):
    print(f"Downloading product with ID: {product_id}...")  # Print message indicating the start of download

    url = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products({product_id})/$value"
    headers = {"Authorization": f"Bearer {access_token}"}
    session = requests.Session()
    session.headers.update(headers)
    response = session.get(url, headers=headers, stream=True)

    if response.status_code == 200:
        file_path = os.path.join(output_dir, f"{product_id}.zip")
        with open(file_path, "wb") as file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    file.write(chunk)
        print(f"Successfully downloaded {product_id}")
    else:
        print(f"Failed to download {product_id}: {response.status_code}")

# Download each product in selected_images
# Refresh the token before each image download
for index, row in selected_images.iterrows():
    zip_path = os.path.join(output_dir, f"{row['Id']}.zip")
    safe_path = os.path.join(output_dir, row['Name'])  # 'Name' already includes the '.SAFE' extension
    if os.path.isfile(zip_path) or os.path.isdir(safe_path):
        print(f"{row['Id']} is already downloaded, skipping.")
        continue
    access_token = get_access_token(config.username, config.password)
    download_product(row['Id'], access_token, output_dir)

## 9. Unzip the downloads

Extract the downloaded `.SAFE` archives and delete the original zip files.

In [ ]:
def unzip_files(folder_path, target_folder=None):
    """
    Unzips all the S1 SLC files in the specified folder and deletes the original zip files.

    :param folder_path: Path to the folder containing the zip files.
    :param target_folder: Path to the folder where the unzipped files will be saved (optional).
    """
    if target_folder is None:
        target_folder = folder_path

    # List all the zip files in the folder
    zip_files = [f for f in os.listdir(folder_path) if f.endswith('.zip')]

    if not zip_files:
        return "Files are already unzipped"

    for zip_file in zip_files:
        zip_path = os.path.join(folder_path, zip_file)

        # Guard against stale/incomplete downloads
        try:
            zip_ref = zipfile.ZipFile(zip_path, 'r')
        except zipfile.BadZipFile:
            print(f"Skipping '{zip_file}': not a valid zip file (likely a stale/incomplete download).")
            continue

        with zip_ref:
            # Avoids re-extracting an archive whose contents are already on disk
            safe_folder_name = zip_ref.namelist()[0].split('/')[0]
            safe_folder_path = os.path.join(target_folder, safe_folder_name)
            if os.path.isdir(safe_folder_path):
                print(f"'{safe_folder_name}' is already unzipped, skipping '{zip_file}'.")
                continue

            # Print message before unzipping
            print(f"Unzipping '{zip_file}'...")
            zip_ref.extractall(target_folder)

        # Delete the original zip file
        #os.remove(zip_path)

        # Print message after unzipping
        print(f"Finished unzipping '{zip_file}'")

    # Print final message
    print("Done processing zip files.")

# Call the function
folder_path = input_dir
unzip_files(folder_path)

## 10. Inspect the input SAFE products

Read each unzipped SAFE product with `esa_snappy`, tabulate its metadata (mode, product type, polarization, bands), and display its quick-look image.

In [ ]:
# Create a list of unzipped Sentinel-1 SAFE directories in the input directory
slc_files = list(iglob(os.path.join(input_dir, '*S1*.SAFE'), recursive=True))
slc_files.sort(key=lambda x: x.split('_')[-5])

# Initialize lists for storing information
name, date, mode, product_type, polarization, band_names = ([] for _ in range(6))

# Create a figure to visualize the input data
fig = plt.figure(figsize=(17, 13))
fig.tight_layout(w_pad=1, h_pad=1)
d = math.ceil(math.sqrt(len(slc_files)))
f = 0

for i in slc_files:
    f += 1
    # Read unzipped product with esa_snappy
    s1_read = esa_snappy.ProductIO.readProduct(i)
    s1_name = s1_read.getName()
    ac_date = re.split('_|T', s1_name)[5]
    date.append(datetime.strftime(datetime.strptime(ac_date, '%Y%m%d'), '%d %b. %Y'))
    name.append(s1_name)
    mode.append(s1_name.split("_")[1])
    product_type.append(s1_name.split("_")[2])
    polarization.append(s1_name.split("_")[4])
    band_names.append(list(s1_read.getBandNames()))
    s1_read.dispose()

    # Visualize a quick-look image available in the SAFE directory
    quick_look_path = os.path.join(i, 'preview', 'quick-look.png')
    if os.path.exists(quick_look_path):
        img = mpimg.imread(quick_look_path)
        ax = fig.add_subplot(d, d, f)
        ax.title.set_text(ac_date)
        ax.imshow(img)

# Create the input data DataFrame
df_slc_data = pd.DataFrame({
    'Name': name,
    'Date': date,
    'Sensing Mode': mode,
    'Product Type': product_type,
    'Polarization': polarization,
    'Available Bands': band_names,
})

# Display the DataFrame
display(df_slc_data)

# Show the figure with quick-look images
plt.show()

## 11. Run the SNAP coherence graph

Auto-detect which IW subswath actually covers the AOI for this orbit, then run the SNAP `gpt` coherence-processing graph twice on the selected triplet (skipping a pair if already processed): once for the pre1/pre2 baseline pair, once for the pre2/post co-event pair, each producing its own terrain-corrected coherence GeoTIFF.

In [ ]:
# Build paths to the selected pre1/pre2/post SAFE products
pre1_row = selected_images.iloc[0]
pre2_row = selected_images.iloc[1]
post_row = selected_images.iloc[2]

# The catalog's 'Name' field already includes the '.SAFE' extension
pre1_safe = os.path.join(input_dir, pre1_row['Name'])
pre2_safe = os.path.join(input_dir, pre2_row['Name'])
post_safe = os.path.join(input_dir, post_row['Name'])

pre1_date_str = pre1_row['AcquisitionDate'].strftime('%Y%m%d')
pre2_date_str = pre2_row['AcquisitionDate'].strftime('%Y%m%d')
post_date_str = post_row['AcquisitionDate'].strftime('%Y%m%d')

def get_subswath_footprint(safe_path, subswath, polarization='vv'):
    pattern = os.path.join(safe_path, 'annotation', f'*-{subswath.lower()}-slc-{polarization}-*.xml')
    matches = glob(pattern)
    if not matches:
        raise FileNotFoundError(f"No {subswath} annotation file found under {safe_path}/annotation")
    points = ET.parse(matches[0]).getroot().findall(
        './/geolocationGrid/geolocationGridPointList/geolocationGridPoint')
    coords = [(float(p.find('longitude').text), float(p.find('latitude').text)) for p in points]
    return shp.geometry.MultiPoint(coords).convex_hull

def select_subswath(safe_path, aoi, subswaths=('IW1', 'IW2', 'IW3')):
    footprints = {sw: get_subswath_footprint(safe_path, sw) for sw in subswaths}
    containing = [sw for sw, fp in footprints.items() if fp.contains(aoi)]
    if containing:
        return containing[0], footprints
    # Fallback if no single subswath fully contains the AOI 
    best = max(footprints, key=lambda sw: footprints[sw].intersection(aoi).area)
    return best, footprints

# All three products share the same relative orbit/track so the subswath choice based on pre1
# applies to pre2 and post as well.
subswath, subswath_footprints = select_subswath(pre1_safe, aoi)
print(f"Auto-selected subswath {subswath} for relative orbit {relative_orbit} "
      f"(checked: {', '.join(subswath_footprints)})")

def process_coherence_pair(safe_a, safe_b, out_path, subswath):
    # Run the SNAP gpt coherence graph on one image pair, skipping it if already processed
    if os.path.isfile(out_path):
        print(colored(f"Coherence product {os.path.basename(out_path)} is already pre-processed!", 'blue'))
        return
    print(colored(f"Pre-processing coherence for {os.path.basename(out_path)}!", 'red'))
    start = datetime.now()
    cmd_parts = ['gpt', dq(graph_file_path),
                 "-PInput1=" + dq(safe_a), "-PInput2=" + dq(safe_b),
                 "-PSubswath=" + dq(subswath),
                 "-PSpeckle_Filter=" + dq(speckle_filter), "-PWindow_size=" + dq(speckle_windowsize),
                 "-PDEM=" + dq(dem), "-PCSR=" + dq(proj),
                 "-PPolygon=" + dq(subset_polygon),
                 "-POutput=" + dq(out_path), "-PFormat=" + dq(out_format)]
    code = execute(' '.join(cmd_parts))  # Execute the command in the terminal
    end = datetime.now()
    # gpt can exit 0 (or the notebook's kernel can be interrupted mid-run) without actually
    # writing the output so also verify the file exists
    if code != 0 or not os.path.isfile(out_path):
        raise RuntimeError(
            f"SNAP gpt failed to produce {out_path} (exit code {code}). "
            "Check the gpt output printed above for the actual error."
        )
    print(colored(f"Coherence product {os.path.basename(out_path)} successfully pre-processed in " + str(end - start) + '!', 'green'))

# Baseline pair (pre1/pre2): expected to show naturally high coherence with no conflict in between
out_path_baseline = os.path.join(prep_dir, f"{pre1_date_str}_{pre2_date_str}_baseline_coh" + suffix)
process_coherence_pair(pre1_safe, pre2_safe, out_path_baseline, subswath)

# Co-event pair (pre2/post): coherence drop relative to the baseline pair is the damage signal
out_path_coevent = os.path.join(prep_dir, f"{pre2_date_str}_{post_date_str}_coevent_coh" + suffix)
process_coherence_pair(pre2_safe, post_safe, out_path_coevent, subswath)

prep_data_baseline = out_path_baseline  # Path to the pre-event baseline coherence product
prep_data_coevent = out_path_coevent    # Path to the co-event coherence product

## 12. Load the processed coherence products

Read the coherence band back from both the baseline-pair and co-event-pair GeoTIFFs with rasterio, and set up the per-triplet results output folder.

In [ ]:
# Load the preprocessed baseline and co-event coherence products, extract each coherence band +
# georeferencing, and set up the per-triplet results output folder
from rasterio.warp import reproject, Resampling, transform_geom
from rasterio.features import geometry_mask

results_dir_triplet = os.path.join(results_dir, f'{pre1_date_str}_{pre2_date_str}_{post_date_str}')
os.makedirs(results_dir_triplet, exist_ok=True)

def read_coherence_band(path):
    with rasterio.open(path) as src:
        coh_band_idx = None
        for i in range(1, src.count + 1):
            finite = src.read(i)
            finite = finite[np.isfinite(finite)]
            if finite.size and finite.min() >= 0 and finite.max() <= 1.01:
                coh_band_idx = i
                break
        if coh_band_idx is None:
            raise ValueError(f"No band in {path} looks like coherence (expected values bounded in [0, 1])")
        data = src.read(coh_band_idx).astype(np.float32)
        profile = src.profile  # CRS/transform
        bounds = src.bounds
        crs = src.crs
    return data, profile, bounds, crs

baseline_coh, coh_profile, coh_bounds, coh_crs = read_coherence_band(prep_data_baseline)
coevent_coh_raw, coevent_profile, _, coevent_crs = read_coherence_band(prep_data_coevent)

# Resample the co-event raster onto the baseline raster's exact grid before any comparison.
coevent_coh = np.zeros_like(baseline_coh)
reproject(
    source=coevent_coh_raw,
    destination=coevent_coh,
    src_transform=coevent_profile['transform'],
    src_crs=coevent_crs,
    dst_transform=coh_profile['transform'],
    dst_crs=coh_crs,
    resampling=Resampling.bilinear,
)

# Build a pixel mask of the true AOI
# detection, saved GeoTIFF/PNG/HTML, summary stats - only reflects the real study area.
aoi_proj_geom = transform_geom('EPSG:4326', coh_crs, shp.geometry.mapping(aoi))
aoi_pixel_mask = geometry_mask([aoi_proj_geom], out_shape=baseline_coh.shape,
                                transform=coh_profile['transform'], invert=True)

# Reproject the raster's bounds to lat/lon for the folium overlays below 
left, bottom, right, top = transform_bounds(coh_crs, 'EPSG:4326', *coh_bounds)
overlay_bounds = [[bottom, left], [top, right]]  

# Ensure oherence difference is only computed where it's meaningful
valid = (baseline_coh > 0) & (coevent_coh > 0) & aoi_pixel_mask
print(f"Loaded baseline pair ({pre1_date_str}-{pre2_date_str}) and co-event pair "
      f"({pre2_date_str}-{post_date_str}): {baseline_coh.shape[1]}x{baseline_coh.shape[0]} px, "
      f"baseline coherence range [{baseline_coh[valid].min():.2f}, {baseline_coh[valid].max():.2f}], "
      f"co-event coherence range [{coevent_coh[valid].min():.2f}, {coevent_coh[valid].max():.2f}] "
      f"over {valid.sum()} valid pixels (clipped to the AOI polygon)")
print(f"Results for this triplet will be saved to: {results_dir_triplet}")

## 13. Detect damage candidates

Threshold the *drop* in coherence between the baseline pair and the co-event pair (the Damage Proxy Map approach) and connectivity-filter it into a damage-candidate mask, then plot both coherence rasters vs. the mask and save the figure to the results folder.

In [ ]:
# Flag pixels where coherence drops relative to the pre-event baseline as damage candidates
coherence_drop_threshold = 0.25
min_connected_pixels = 8

coherence_drop = baseline_coh - coevent_coh  # positive = coherence dropped after the baseline pair
damage_candidate = valid & (coherence_drop > coherence_drop_threshold)
labeled, num_features = ndimage.label(damage_candidate, structure=np.ones((3, 3)))
sizes = ndimage.sum(damage_candidate, labeled, range(1, num_features + 1))
keep_labels = np.where(sizes >= min_connected_pixels)[0] + 1
damage_mask = np.isin(labeled, keep_labels)

valid_pixels = valid.sum()
damage_pixels = damage_mask.sum()
print(f"Damage candidates: {damage_pixels / valid_pixels:.1%} of the valid AOI coverage "
      f"({damage_pixels} of {valid_pixels} pixels, coherence drop > {coherence_drop_threshold})")

# Static side-by-side view: baseline coherence vs. co-event coherence vs. the derived damage mask
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
im0 = axes[0].imshow(np.ma.masked_where(~valid, baseline_coh), cmap='Spectral', vmin=0, vmax=1)
axes[0].set_title(f'Baseline coherence ({pre1_date_str} - {pre2_date_str})')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, label='Coherence')

im1 = axes[1].imshow(np.ma.masked_where(~valid, coevent_coh), cmap='Spectral', vmin=0, vmax=1)
axes[1].set_title(f'Co-event coherence ({pre2_date_str} - {post_date_str})')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, label='Coherence')

axes[2].imshow(np.ma.masked_where(~valid, coevent_coh), cmap='gray', vmin=0, vmax=1)
axes[2].imshow(np.ma.masked_where(~damage_mask, damage_mask), cmap='autumn_r', alpha=0.8)
axes[2].set_title('Damage candidates (coherence drop vs. baseline)')
axes[2].axis('off')
plt.tight_layout()

fig.savefig(os.path.join(results_dir_triplet, 'coherence_damage_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 14. Build the interactive damage map

Render the baseline coherence, co-event coherence, and damage-candidate mask as toggleable overlays on a folium map, and save it as HTML.

In [ ]:
# Interactive damage map: baseline coherence, co-event coherence, and damage-candidate mask over the AOI
def to_rgba(data, mask, cmap_name, vmin=0, vmax=1, alpha=200):
    norm = np.clip((data - vmin) / (vmax - vmin), 0, 1)
    rgba = (plt.colormaps[cmap_name](norm) * 255).astype(np.uint8)
    rgba[..., 3] = np.where(mask, alpha, 0)
    return rgba

baseline_rgba = to_rgba(baseline_coh, valid, 'Spectral')
coevent_rgba = to_rgba(coevent_coh, valid, 'Spectral')
damage_rgba = np.zeros((*damage_mask.shape, 4), dtype=np.uint8)
damage_rgba[..., 0] = 255  # solid red
damage_rgba[..., 3] = np.where(damage_mask, 220, 0)

damage_map_fig = folium.Figure(width=2000, height=1000)
damage_map = folium.Map([aoi_centroid.y, aoi_centroid.x],
                         zoom_start=12,
                         tiles='cartodbdark_matter').add_to(damage_map_fig)

folium.GeoJson(data=gpd.GeoSeries(aoi).to_json(),
               style_function=lambda x: {'color': 'white', 'weight': 1, 'fillOpacity': 0}).add_to(damage_map)
folium.raster_layers.ImageOverlay(baseline_rgba, bounds=overlay_bounds,
                                   name=f'Baseline coherence {pre1_date_str}-{pre2_date_str}',
                                   opacity=0.85, show=False).add_to(damage_map)
folium.raster_layers.ImageOverlay(coevent_rgba, bounds=overlay_bounds,
                                   name=f'Co-event coherence {pre2_date_str}-{post_date_str}',
                                   opacity=0.85).add_to(damage_map)
folium.raster_layers.ImageOverlay(damage_rgba, bounds=overlay_bounds,
                                   name='Damage candidates',
                                   opacity=0.85).add_to(damage_map)
folium.LayerControl().add_to(damage_map)

# Save Folium map and open it manually.
damage_map_path = os.path.abspath(os.path.join(results_dir_triplet, 'damage_map.html'))
damage_map.save(damage_map_path)
print(f"Map saved. Open it in your browser from: {damage_map_path}")

## 15. Save georeferenced outputs

Write the final 3-band GeoTIFF (baseline coherence, co-event coherence, damage mask) and a JSON summary of the damage statistics to the results folder.

In [ ]:
# Georeferenced results: a 3-band GeoTIFF (baseline coherence, co-event coherence, damage mask)
output_profile = coh_profile.copy()
output_profile.update(count=3, dtype='float32', nodata=0)

output_tif_path = os.path.join(results_dir_triplet, 'damage_assessment.tif')
with rasterio.open(output_tif_path, 'w', **output_profile) as dst:
    dst.write(baseline_coh, 1)
    dst.write(coevent_coh, 2)
    dst.write(damage_mask.astype(np.float32), 3)
    dst.set_band_description(1, 'baseline_coherence')
    dst.set_band_description(2, 'coevent_coherence')
    dst.set_band_description(3, 'damage_candidate')

summary = {
    'pre1_acquisition_date': pre1_date_str,
    'pre2_acquisition_date': pre2_date_str,
    'post_acquisition_date': post_date_str,
    'relative_orbit': int(pre1_row['RelativeOrbitNumber']),
    'coherence_drop_threshold': coherence_drop_threshold,
    'min_connected_pixels': min_connected_pixels,
    'valid_pixels': int(valid_pixels),
    'damage_candidate_pixels': int(damage_pixels),
    'damage_fraction': float(damage_pixels / valid_pixels),
}
with open(os.path.join(results_dir_triplet, 'summary_stats.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Saved damage_assessment.tif, coherence_damage_comparison.png, damage_map.html, and "
      f"summary_stats.json to {results_dir_triplet}")